# Kaggle Pre-training: Continued MLM on Log Corpus

Continued masked-language-model (MLM) pre-training of `cisco-ai/SecureBERT2.0-base` on the log corpus reconstructed from HDFS TraceBench. The base SecureBERT 2.0 was pre-trained on cybersecurity-domain *prose* (threat reports, CVEs, CTI bulletins). Production logs are a different surface form (template-driven, repetitive, structured). One epoch of continued MLM on logs adapts the encoder to log syntax cheaply and benefits every downstream head (tier-2 classifier, NER, cross-encoder, biencoder).

Output is a HuggingFace model directory drop-in for `--model-id` in `training/train_transformer.py`, `kaggle_train_ner.ipynb`, and `kaggle_train_cross_encoder.ipynb`.

Expected runtime on Kaggle Free GPU (T4) is under the 9h session limit. Enable a GPU accelerator before running training cells. Use bf16 if the kernel image supports it; fp16 otherwise.

## 1. Locate the repo

Run this notebook from the project root when possible. If Kaggle places the repo inside `/kaggle/working`, the cell below tries to find it.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = Path.cwd()
GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'

if not (REPO_DIR / 'training' / 'train.py').exists():
    matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
    if matches:
        REPO_DIR = matches[0].parents[1]
    else:
        clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
        if not clone_dir.exists():
            subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
        REPO_DIR = clone_dir

if not (REPO_DIR / 'training' / 'train.py').exists():
    raise FileNotFoundError('Could not find training/train.py. Upload or clone the repo into /kaggle/working first.')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)

## 2. Install training dependencies

Kaggle already includes PyTorch. We install the transformer/datasets/accelerate stack used here. ONNX is optional for MLM (the artifact is consumed as a HuggingFace directory by downstream notebooks).

In [ ]:
%pip install -q --upgrade-strategy only-if-needed datasets

## 3. Attach the HDFS TraceBench dataset

The same preprocessed files used by tier-1/tier-2 (`HDFS_v3_TraceBench/preprocessed/normal_trace.csv`, `failure_trace.csv`, `eventId.json`) supply the unlabeled text corpus. If attached as a Kaggle dataset, this cell symlinks the input path into the expected project location.

In [ ]:
PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'

def has_trace_files(path: Path) -> bool:
    return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

if not has_trace_files(PREPROCESSED_DIR):
    input_root = Path('/kaggle/input')
    candidates = [p for p in input_root.rglob('preprocessed') if has_trace_files(p)]
    if not candidates:
        # Datasets may publish trace files at the top level (no nested 'preprocessed' dir).
        candidates = [p for p in input_root.rglob('*') if p.is_dir() and has_trace_files(p)]
    if not candidates:
        raise FileNotFoundError('No Kaggle input folder with normal_trace.csv and failure_trace.csv was found.')
    source = candidates[0]
    PREPROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PREPROCESSED_DIR.exists() and not PREPROCESSED_DIR.is_symlink():
        raise FileExistsError(f'{PREPROCESSED_DIR} exists but does not contain the expected trace files.')
    if not PREPROCESSED_DIR.exists():
        os.symlink(source, PREPROCESSED_DIR, target_is_directory=True)
    print('Linked dataset:', source, '->', PREPROCESSED_DIR)
else:
    print('Dataset already available:', PREPROCESSED_DIR)

## 4. Build the unlabeled MLM corpus

We reuse `training.text_dataset.build_windows` to reconstruct multi-line log windows from the count vectors, then drop the labels — MLM is self-supervised. Both normal and failure tasks contribute, since MLM benefits from the full vocabulary distribution. Sanity-print one window before training.

In [ ]:
from training.text_dataset import build_windows

texts, _, stats = build_windows(sample_normal=2, sample_failure=2, random_state=42)
print(json.dumps(stats, indent=2))
for i, t in enumerate(texts):
    print(f'\n--- preview window {i}, chars={len(t)} ---')
    print(t[:1000])
    if len(t) > 1000:
        print(f'... [+{len(t) - 1000} more chars]')

## 5. Sampled MLM run (verify environment first)

Verify tokenizer/model download, GPU training, and checkpointing on a small subset (5K normal + 1K failure windows, 1 epoch) before committing a full session. Output: `models/securebert2-logs-mlm-sample/`.

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

MODEL_ID = 'cisco-ai/SecureBERT2.0-base'
MAX_LENGTH = 1024
MLM_PROB = 0.15

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

print('CUDA:', torch.cuda.is_available(), 'bf16:', use_bf16, 'fp16:', use_fp16)
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)

sample_texts, _, sample_stats = build_windows(
    sample_normal=5000, sample_failure=1000, random_state=42
)
print('sampled texts:', len(sample_texts), sample_stats)

ds = Dataset.from_dict({'text': sample_texts})

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True,
    )

ds_tok = ds.map(tokenize, batched=True, remove_columns=['text'])
ds_split = ds_tok.train_test_split(test_size=0.05, seed=42)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=MLM_PROB
)

sample_out = REPO_DIR / 'models' / 'securebert2-logs-mlm-sample'
args = TrainingArguments(
    output_dir=str(sample_out),
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    bf16=use_bf16,
    fp16=use_fp16,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=50,
    save_total_limit=1,
    report_to='none',
    dataloader_num_workers=2,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=ds_split['train'],
    eval_dataset=ds_split['test'],
)
trainer.train()
metrics = trainer.evaluate()
print('sampled eval:', metrics)

trainer.save_model(str(sample_out / 'final'))
tokenizer.save_pretrained(str(sample_out / 'final'))
(sample_out / 'final' / 'mlm_metrics.json').write_text(json.dumps(metrics, indent=2))
print('saved:', sample_out / 'final')

## 6. Full MLM run (uncomment when sampled run succeeds)

Pre-train on the entire HDFS TraceBench reconstructed corpus for 1 epoch. Continued pre-training does not need many epochs — one pass is the standard recipe and avoids catastrophic forgetting of Cisco's security-domain pretraining. Output: `models/securebert2-logs-mlm/final/`. Estimated runtime ~4-6h on Kaggle T4 with bf16/fp16.

In [ ]:
# texts_full, _, stats_full = build_windows(random_state=42)
# print('full corpus:', len(texts_full), stats_full)
#
# ds_full = Dataset.from_dict({'text': texts_full}).map(
#     tokenize, batched=True, remove_columns=['text']
# )
# ds_full_split = ds_full.train_test_split(test_size=0.02, seed=42)
#
# full_out = REPO_DIR / 'models' / 'securebert2-logs-mlm'
# args_full = TrainingArguments(
#     output_dir=str(full_out),
#     num_train_epochs=1,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     gradient_accumulation_steps=4,
#     learning_rate=5e-5,
#     weight_decay=0.01,
#     warmup_ratio=0.05,
#     bf16=use_bf16,
#     fp16=use_fp16,
#     save_strategy='steps',
#     save_steps=1000,
#     eval_strategy='steps',
#     eval_steps=1000,
#     logging_steps=200,
#     save_total_limit=2,
#     report_to='none',
#     dataloader_num_workers=2,
#     seed=42,
# )
# trainer_full = Trainer(
#     model=AutoModelForMaskedLM.from_pretrained(MODEL_ID),
#     args=args_full,
#     data_collator=collator,
#     train_dataset=ds_full_split['train'],
#     eval_dataset=ds_full_split['test'],
# )
# trainer_full.train()
# full_metrics = trainer_full.evaluate()
# trainer_full.save_model(str(full_out / 'final'))
# tokenizer.save_pretrained(str(full_out / 'final'))
# (full_out / 'final' / 'mlm_metrics.json').write_text(json.dumps(full_metrics, indent=2))
# print('full eval:', full_metrics)

## 7. Inspect artifacts

Confirm the final model directory has the expected HuggingFace files.

In [ ]:
for candidate in [REPO_DIR / 'models' / 'securebert2-logs-mlm', REPO_DIR / 'models' / 'securebert2-logs-mlm-sample']:
    final = candidate / 'final'
    if not final.exists():
        continue
    print(f'=== {final} ===')
    for path in sorted(final.glob('*')):
        if path.is_file():
            print(f'  {path.name}: {path.stat().st_size:,} bytes')
    metrics_path = final / 'mlm_metrics.json'
    if metrics_path.exists():
        print('  metrics:', json.loads(metrics_path.read_text()))

## 8. Package artifacts

Zip the final model directory into `/kaggle/working/` so it appears in Kaggle's notebook outputs. Prefer the full-run artifact when available.

In [ ]:
import shutil

full_dir = REPO_DIR / 'models' / 'securebert2-logs-mlm' / 'final'
sample_dir = REPO_DIR / 'models' / 'securebert2-logs-mlm-sample' / 'final'
src_dir = full_dir if full_dir.exists() else sample_dir
tag = 'full' if src_dir is full_dir else 'sample'

if not src_dir.exists():
    raise FileNotFoundError('No MLM artifact found. Run the sampled or full training cell first.')

out_zip = KAGGLE_WORKING / f'logfilter-mlm-{tag}-artifacts'
shutil.make_archive(str(out_zip), 'zip', root_dir=src_dir.parent.parent, base_dir=Path(*src_dir.parts[-3:]).as_posix())
print('packaged:', out_zip.with_suffix('.zip'))

## 9. Output description + how to consume in repo

Download `/kaggle/working/logfilter-mlm-full-artifacts.zip` (or `-sample-` for the verification run) and extract it at the repository root. The archive contains the HuggingFace model directory `models/securebert2-logs-mlm/final/` with `config.json`, `model.safetensors`, tokenizer files, and `mlm_metrics.json`.

To use this domain-adapted base in downstream training, point the relevant `--model-id` (or notebook constant) at the local path:

```bash
python training/train_transformer.py \
    --model-id models/securebert2-logs-mlm/final \
    --epochs 2 \
    --output-dir models/tier2
```

The same path also serves `kaggle_train_ner.ipynb` and `kaggle_train_cross_encoder.ipynb` as the encoder backbone. This is the foundation step — every downstream classifier/NER/cross-encoder benefits from log-domain pre-training before its head is fine-tuned.